# Week4
## Kilosort-Function Fixed-Template Sorting on Compressed-Reconstructed Data

Goal: use Kilosort detection/matching functions on compressed-reconstructed data, while keeping baseline templates fixed (no template re-learning).


In [34]:
import numpy as np
from pathlib import Path
import inspect
import importlib

ROOT = Path('F:/academic')
DATA_ROOT = ROOT / '.test_data'
BASE_RESULTS = DATA_ROOT / 'kilosort4'

compressed_bin = ROOT / 'whitened_recon_ratio_0.10.bin'
if not compressed_bin.exists():
    compressed_bin = ROOT / 'whitened_recon_ratio_0.20.bin'
if not compressed_bin.exists():
    raise FileNotFoundError('Missing whitened_recon_ratio_*.bin')

print('Compressed bin:', compressed_bin)


Compressed bin: F:\academic\whitened_recon_ratio_0.10.bin


In [35]:
# Load baseline outputs from uncompressed run
templates_base = np.load(BASE_RESULTS / 'templates.npy')
ops_base = np.load(BASE_RESULTS / 'ops.npy', allow_pickle=True).item()
st_base = np.load(BASE_RESULTS / 'spike_times.npy').squeeze().astype(np.int64)
clu_base = np.load(BASE_RESULTS / 'spike_clusters.npy').squeeze().astype(np.int64)

print('templates_base shape:', templates_base.shape)
print('ops keys example:', list(ops_base.keys())[:12])

# Project-specific bin format from prior checks
data_dtype = np.float32
src_n_chan = 383
raw = np.memmap(compressed_bin, dtype=data_dtype, mode='r')
n_samples = raw.size // src_n_chan
data_cmp = raw[: n_samples * src_n_chan].reshape(n_samples, src_n_chan)
print('compressed data shape:', data_cmp.shape)


templates_base shape: (267, 61, 383)
ops keys example: ['n_chan_bin', 'fs', 'batch_size', 'nblocks', 'Th_universal', 'Th_learned', 'tmin', 'tmax', 'nt', 'shift', 'scale', 'batch_downsampling']
compressed data shape: (1350000, 383)


### Note: 383 vs 385 channels
Baseline Kilosort outputs often use total-channel probe definitions (e.g., 385 including extra/sync/reference channels).
Your reconstructed compressed data here is 383-channel float32.
For fixed-template matching, we only use channels shared by both arrays (template channels clipped to data channels).


In [36]:
# Discover Kilosort internal modules/functions in your environment
ks_spk = importlib.import_module('kilosort.spikedetect')
ks_tm = importlib.import_module('kilosort.template_matching')

print('kilosort.spikedetect path:', ks_spk.__file__)
print('kilosort.template_matching path:', ks_tm.__file__)

spk_fns = [n for n,v in inspect.getmembers(ks_spk, inspect.isfunction)]
tm_fns = [n for n,v in inspect.getmembers(ks_tm, inspect.isfunction)]
print('spikedetect functions:', spk_fns)
print('template_matching functions:', tm_fns)


kilosort.spikedetect path: E:\Anaconda\envs\kilosort\Lib\site-packages\kilosort\spikedetect.py
kilosort.template_matching path: E:\Anaconda\envs\kilosort\Lib\site-packages\kilosort\template_matching.py
spikedetect functions: ['extract_snippets', 'extract_wPCA_wTEMP', 'get_waves', 'log_performance', 'max_pool1d', 'max_pool2d', 'my_max2d', 'my_sum2d', 'nearest_chans', 'run', 'template_centers', 'template_match', 'template_path', 'yweighted']
template_matching functions: ['align_U', 'extract', 'log_performance', 'max_pool1d', 'max_pool2d', 'merging_function', 'postprocess_templates', 'prepare_extract', 'prepare_matching', 'roll_features', 'run_matching']


In [37]:
# Build fixed-template inputs for Kilosort matching
n_templates, nt, n_chan_tpl = templates_base.shape
n_use = min(n_chan_tpl, data_cmp.shape[1])
templates_fixed = templates_base[:, :, :n_use].astype(np.float32, copy=False)

# Candidate spike times from baseline timeline (strict time alignment mode)
half = nt // 2
valid = (st_base - half >= 0) & (st_base + (nt - half) <= n_samples)
st_candidates = st_base[valid]
clu_ref = clu_base[valid]

print('templates_fixed:', templates_fixed.shape)
print('candidate spikes:', st_candidates.shape[0])


templates_fixed: (267, 61, 383)
candidate spikes: 137378


### Kilosort Fixed-Template Matching (prepare_matching + run_matching)
This uses Kilosort functions directly and keeps baseline templates fixed.
No template re-learning is performed.


In [38]:
import gc
import torch
from kilosort import template_matching as ks_tm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# GPU cleanup: previous notebook runs may keep large tensors alive.
for _name in ['U', 'ctc', 'X', 'tpl_t', 'wPCA_t', 'B', 'Cf']:
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass
    free_b, total_b = torch.cuda.mem_get_info()
    print(f'GPU mem before alloc: free={free_b/1e9:.2f} GB / total={total_b/1e9:.2f} GB')

# Build ops for matching (copy baseline ops and override only required fields)
ops_tm = dict(ops_base)
if 'settings' in ops_tm and isinstance(ops_tm['settings'], dict):
    ops_tm['settings'] = dict(ops_tm['settings'])
else:
    ops_tm['settings'] = {}

ops_tm['nt'] = int(nt)
ops_tm['nt0min'] = int(ops_base.get('nt0min', nt//2))
ops_tm['Th_learned'] = float(ops_base.get('Th_learned', 8))
ops_tm['max_peels'] = int(ops_base.get('max_peels', 10))

# Ensure wPCA exists and shape is [n_pcs, nt]
wPCA = ops_base.get('wPCA', None)
if wPCA is None:
    raise RuntimeError('ops_base does not contain wPCA; cannot use template_matching.run_matching path.')

wPCA_np = np.asarray(wPCA, dtype=np.float32)
print('wPCA_np shape:', wPCA_np.shape, 'nbytes:', wPCA_np.nbytes)
wPCA_t = torch.from_numpy(wPCA_np).to(device=device, non_blocking=True).contiguous()
ops_tm['wPCA'] = wPCA_t

# U required by run_matching has shape [n_units, n_pcs, n_channels]
tpl_t = torch.as_tensor(templates_fixed, dtype=torch.float32, device=device)
U = torch.einsum('utc,pt->upc', tpl_t, wPCA_t).contiguous()
del tpl_t

if device.type == 'cuda':
    free_b, total_b = torch.cuda.mem_get_info()
    print(f'GPU mem after U alloc: free={free_b/1e9:.2f} GB / total={total_b/1e9:.2f} GB')

ctc = ks_tm.prepare_matching(ops_tm, U)
print('wPCA:', tuple(wPCA_t.shape), 'U:', tuple(U.shape), 'ctc:', tuple(ctc.shape))


device: cuda


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Run matching on compressed data with fixed time windows (GPU-memory safe)
half = nt // 2
valid = (st_base - half >= 0) & (st_base + (nt - half) <= n_samples)
st_candidates = st_base[valid]
clu_ref = clu_base[valid]

pred = np.full(st_candidates.shape[0], -1, dtype=np.int64)
ampv = np.full(st_candidates.shape[0], np.nan, dtype=np.float32)
thv = np.full(st_candidates.shape[0], np.nan, dtype=np.float32)

win = 20000  # samples per GPU batch
tol = 12

for t0 in range(0, n_samples, win):
    t1 = min(n_samples, t0 + win)
    mask = (st_candidates >= t0 + half) & (st_candidates < t1 - half)
    if not np.any(mask):
        continue

    idx = np.where(mask)[0]
    X_np = data_cmp[t0:t1, :n_use].T.astype(np.float32, copy=False)
    X = torch.as_tensor(X_np, dtype=torch.float32, device=device).contiguous()

    with torch.no_grad():
        stt, amps, th_amps, _ = ks_tm.run_matching(ops_tm, X, U, ctc, device=device)

    abs_t = stt[:,0].detach().cpu().numpy().astype(np.int64) + t0
    loc_k = stt[:,1].detach().cpu().numpy().astype(np.int64)
    amps_np = amps.detach().cpu().numpy().squeeze()
    th_np = th_amps.detach().cpu().numpy().squeeze()

    t_idx = st_candidates[idx]
    if len(abs_t) > 0:
        pos = np.searchsorted(abs_t, t_idx)
        left = np.clip(pos-1, 0, len(abs_t)-1)
        right = np.clip(pos, 0, len(abs_t)-1)
        dl = np.abs(t_idx - abs_t[left])
        dr = np.abs(t_idx - abs_t[right])
        use_l = dl <= dr
        nn = np.where(use_l, left, right)
        d = np.abs(t_idx - abs_t[nn])
        ok = d <= tol

        pred[idx[ok]] = loc_k[nn[ok]]
        ampv[idx[ok]] = amps_np[nn[ok]]
        thv[idx[ok]] = th_np[nn[ok]]

    del X
    if device.type == 'cuda':
        torch.cuda.empty_cache()

clu_cmp_fixed = pred
amp_cmp_fixed = ampv
th_cmp_fixed = thv
st_cmp_fixed = st_candidates

matched = clu_cmp_fixed >= 0
print('candidate spikes:', st_cmp_fixed.shape[0])
print('matched by Kilosort run_matching:', int(matched.sum()), f'({matched.mean()*100:.2f}%)')
if matched.any():
    acc = (clu_cmp_fixed[matched] == clu_ref[matched]).mean()
    print(f'label agreement on matched: {acc*100:.2f}%')


In [ ]:
out = {
    'compressed_bin': str(compressed_bin),
    'method': 'kilosort_template_matching_run_matching_fixed_baseline_templates',
    'st': st_cmp_fixed,
    'clu': clu_cmp_fixed,
    'amp': amp_cmp_fixed,
    'th_amp': th_cmp_fixed,
    'baseline_clu_at_same_time': clu_ref,
    'template_shape': templates_fixed.shape,
    'compressed_shape': data_cmp.shape,
}
np.save(ROOT / 'week4_kilosort_function_fixed_template_results.npy', out, allow_pickle=True)
print('saved:', ROOT / 'week4_kilosort_function_fixed_template_results.npy')
